In [ ]:
import sys
import subprocess

print("Python:", sys.version)

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--upgrade",
    "--only-binary=:all:",
    "transformers==4.57.6",
    "tokenizers==0.22.2",
    "datasets==4.5.0",
    "huggingface-hub>=0.34,<1.0",
    "Pillow>=11.1,<12",
    "pandas",
    "matplotlib",
    "tqdm",
])

print("\nInstallation complete.")
print("Now select Runtime > Restart session, then run Block 2.")

In [ ]:
import os
import time
import json
import random
import re
import io
from pathlib import Path
from collections import Counter

os.environ["TOKENIZERS_PARALLELISM"] = "false"
START = time.monotonic()

import torch
import transformers
import datasets
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from datasets import load_dataset, Image as DatasetImage
from torch.utils.data import DataLoader
from transformers import BlipProcessor, BlipForQuestionAnswering
from tqdm.auto import tqdm
from IPython.display import display


# ================= SETTINGS =================

SEED = 42
TRAIN_SIZE = 800
EVAL_SIZE = 100

BATCH_SIZE = 4
EPOCHS = 15
LEARNING_RATE = 2e-5
MAX_TRAIN_MINUTES = 30

MODEL_ID = "Salesforce/blip-vqa-base"
DATASET_ID = "Raluck/vizwiz-vqa-en"

ROOT = Path("/content/visionbridge_t4")
EXPORT_DIR = ROOT / "export"
MODEL_DIR = EXPORT_DIR / "blip_vizwiz"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), (
    "GPU unavailable. Select Runtime > Change runtime type > T4 GPU."
)

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
    "GB",
)


def elapsed_minutes():
    return (time.monotonic() - START) / 60


# ================= ANSWER PROCESSING =================

def normalize_answer(text):
    # Lightweight normalization for diagnostic evaluation.
    # This is not the full official VQA normalization.
    text = re.sub(r"[^a-z0-9 ]", " ", str(text).lower())
    return " ".join(text.split())


def get_references(answers):
    if isinstance(answers, dict):
        return [str(answer) for answer in answers["answer"]]

    return [
        str(answer["answer"] if isinstance(answer, dict) else answer)
        for answer in answers
    ]


def agreement_score(prediction, references):
    # Average agreement across leave-one-annotator-out reference sets.
    prediction = normalize_answer(prediction)
    references = [normalize_answer(a) for a in references]

    if not references:
        return 0.0

    scores = []
    for excluded_index in range(len(references)):
        matches = sum(
            prediction == answer
            for index, answer in enumerate(references)
            if index != excluded_index
        )
        scores.append(min(1.0, matches / 3.0))

    return sum(scores) / len(scores)


# ================= STREAM DATA SUBSETS =================

def prepare_subset(split, requested_size):
    print(f"\nPreparing {split}: {requested_size} examples")

    # Streaming avoids downloading the full VizWiz image archives.
    stream = load_dataset(
        DATASET_ID,
        split=split,
        streaming=True,
    )

    # Decode images ourselves so errors can be handled per record.
    stream = stream.cast_column(
        "image",
        DatasetImage(decode=False),
    )

    # Small shuffle buffer keeps download/memory costs low.
    # This is a bounded stream sample, not a uniform full-dataset sample.
    stream = stream.shuffle(seed=SEED, buffer_size=64)

    image_dir = ROOT / split
    image_dir.mkdir(parents=True, exist_ok=True)

    records = []
    skipped = 0

    progress = tqdm(
        total=requested_size,
        desc=f"Preparing {split}",
    )

    for row in stream:
        if elapsed_minutes() > 20:
            progress.close()
            raise RuntimeError(
                "Data preparation exceeded 20 minutes. "
                "Reduce TRAIN_SIZE and EVAL_SIZE, then rerun."
            )

        try:
            raw_image = row["image"]
            source = (
                io.BytesIO(raw_image["bytes"])
                if raw_image.get("bytes")
                else raw_image["path"]
            )

            image_name = row.get(
                "image_name",
                f"{split}_{len(records):06d}.jpg",
            )
            image_path = image_dir / Path(image_name).name

            with Image.open(source) as image:
                image = image.convert("RGB")
                image.thumbnail((640, 640))
                image.save(image_path, format="JPEG", quality=90)

            references = get_references(row["answers"])
            normalized = [
                normalize_answer(answer)
                for answer in references
                if normalize_answer(answer)
            ]

            if not normalized:
                raise ValueError("No usable reference answers")

            # Use the most common human answer as the training target.
            target = Counter(normalized).most_common(1)[0][0]

            records.append({
                "image_path": str(image_path),
                "image_name": image_name,
                "question": str(row["question"]),
                "references": references,
                "target": target,
                "answerable": int(row["answerable"]),
            })
            progress.update(1)

        except (OSError, ValueError, KeyError, IndexError) as error:
            skipped += 1
            if skipped <= 3:
                print("Skipped record:", error)

        if len(records) >= requested_size:
            break

    progress.close()

    if len(records) < requested_size:
        raise RuntimeError(
            f"Only {len(records)} usable examples in {split}; "
            f"requested {requested_size}."
        )

    print(
        f"{split}: {len(records)} ready, {skipped} skipped. "
        f"Elapsed: {elapsed_minutes():.1f} minutes"
    )
    return records


train_records = prepare_subset("train", TRAIN_SIZE)
eval_records = prepare_subset("validation", EVAL_SIZE)

train_names = {row["image_name"] for row in train_records}
eval_names = {row["image_name"] for row in eval_records}

assert not (train_names & eval_names), "Train/validation overlap detected."

with open(EXPORT_DIR / "subset_manifest.json", "w") as file:
    json.dump({
        "dataset": DATASET_ID,
        "seed": SEED,
        "sampling": "streaming with shuffle buffer of 64",
        "train_images": sorted(train_names),
        "validation_images": sorted(eval_names),
    }, file, indent=2)


# ================= LOAD BLIP =================

print("\nLoading BLIP...")

processor = BlipProcessor.from_pretrained(MODEL_ID)
model = BlipForQuestionAnswering.from_pretrained(
    MODEL_ID,
).to("cuda")

# Partial fine-tuning keeps the workload small.
# Freeze the image encoder, question encoder, and most decoder layers.
model.requires_grad_(False)

for layer in model.text_decoder.bert.encoder.layer[-2:]:
    layer.requires_grad_(True)

model.text_decoder.cls.requires_grad_(True)

trainable_parameters = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]

print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in trainable_parameters):,}",
)
print(f"Elapsed: {elapsed_minutes():.1f} minutes")


# ================= BATCH PREPROCESSING =================

def make_batch(records, training=True):
    images = []

    for row in records:
        with Image.open(row["image_path"]) as image:
            images.append(image.convert("RGB"))

    batch = processor(
        images=images,
        text=[row["question"] for row in records],
        padding=True,
        truncation=True,
        max_length=48,
        return_tensors="pt",
    )

    if training:
        answer_tokens = processor.tokenizer(
            [row["target"] for row in records],
            padding=True,
            truncation=True,
            max_length=24,
            return_tensors="pt",
        )

        decoder_ids = answer_tokens.input_ids.clone()
        decoder_ids[:, 0] = model.config.text_config.bos_token_id

        labels = decoder_ids.clone()
        labels[answer_tokens.attention_mask == 0] = -100

        # Separate decoder inputs prevent embedding the -100 loss mask.
        batch["decoder_input_ids"] = decoder_ids
        batch["decoder_attention_mask"] = answer_tokens.attention_mask
        batch["labels"] = labels

    return {
        name: tensor.to("cuda")
        for name, tensor in batch.items()
    }


@torch.inference_mode()
def generate_predictions(records):
    model.eval()
    predictions = []

    for start in tqdm(
        range(0, len(records), BATCH_SIZE),
        desc="Generating answers",
    ):
        batch_records = records[start:start + BATCH_SIZE]
        inputs = make_batch(batch_records, training=False)

        with torch.autocast("cuda", dtype=torch.float16):
            generated = model.generate(
                **inputs,
                max_new_tokens=16,
                num_beams=1,
                do_sample=False,
            )

        predictions.extend(
            processor.batch_decode(
                generated,
                skip_special_tokens=True,
            )
        )

    return [prediction.strip() for prediction in predictions]


# ================= BASELINE EVALUATION =================

print("\nEvaluating pretrained baseline...")

baseline_predictions = generate_predictions(eval_records)

baseline_scores = [
    agreement_score(prediction, row["references"])
    for prediction, row in zip(
        baseline_predictions,
        eval_records,
    )
]

print(
    "Baseline diagnostic agreement:",
    f"{100 * sum(baseline_scores) / len(baseline_scores):.2f}%",
)


# ================= TRAINING =================

train_loader = DataLoader(
    train_records,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=make_batch,
)

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=0.01,
)

scaler = torch.amp.GradScaler("cuda")

training_start = time.monotonic()

# Cap training at 25 minutes, and stop by minute 45 from Block 2 start.
training_deadline = min(
    training_start + MAX_TRAIN_MINUTES * 60,
    START + 45 * 60,
)

history = []
optimizer_updates = 0
timer_stopped = False

print("\nStarting training...")

for epoch in range(EPOCHS):
    model.train()

    # Frozen encoders remain in evaluation mode.
    model.vision_model.eval()
    model.text_encoder.eval()

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
    )

    for batch in progress:
        if time.monotonic() >= training_deadline:
            timer_stopped = True
            break

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(**batch).loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                "Non-finite training loss. "
                "Lower LEARNING_RATE and rerun."
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            trainable_parameters,
            max_norm=1.0,
        )

        previous_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()

        # A decreased scale indicates an overflow-skipped optimizer step.
        if scaler.get_scale() >= previous_scale:
            optimizer_updates += 1

        loss_value = loss.item()
        history.append({
            "epoch": epoch + 1,
            "batch": len(history) + 1,
            "loss": loss_value,
            "elapsed_minutes": elapsed_minutes(),
        })

        progress.set_postfix(
            loss=f"{loss_value:.3f}",
            minutes=f"{elapsed_minutes():.1f}",
        )

    if timer_stopped:
        print("\nTraining time limit reached.")
        break

assert optimizer_updates > 0, (
    "No optimizer updates completed. "
    "The checkpoint has not been trained."
)


# ================= SAVE TRAINED CHECKPOINT =================

print("\nSaving trained model...")

model.eval()
model.save_pretrained(
    MODEL_DIR,
    safe_serialization=True,
)
processor.save_pretrained(MODEL_DIR)

pd.DataFrame(history).to_csv(
    EXPORT_DIR / "training_loss.csv",
    index=False,
)

# Save before evaluation so weights remain available if evaluation fails.
training_info = {
    "model": MODEL_ID,
    "dataset": DATASET_ID,
    "train_size": len(train_records),
    "eval_size": len(eval_records),
    "epochs_requested": EPOCHS,
    "optimizer_updates": optimizer_updates,
    "timer_stopped": timer_stopped,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "fine_tuning": "last two BLIP decoder layers and language-model head",
}

with open(EXPORT_DIR / "training_info.json", "w") as file:
    json.dump(training_info, file, indent=2)

print("Saved model:", MODEL_DIR)


# ================= TRAINED MODEL EVALUATION =================

print("\nEvaluating trained model...")

trained_predictions = generate_predictions(eval_records)

results = pd.DataFrame([
    {
        "image": row["image_name"],
        "question": row["question"],
        "answerable": row["answerable"],
        "references": " | ".join(row["references"]),
        "baseline_prediction": baseline,
        "trained_prediction": trained,
        "baseline_score": agreement_score(
            baseline, row["references"]
        ),
        "trained_score": agreement_score(
            trained, row["references"]
        ),
    }
    for row, baseline, trained in zip(
        eval_records,
        baseline_predictions,
        trained_predictions,
    )
])

results.to_csv(
    EXPORT_DIR / "predictions.csv",
    index=False,
)

summary = {
    **training_info,
    "metric": "diagnostic VQA agreement with approximate normalization",
    "baseline_agreement": float(results["baseline_score"].mean()),
    "trained_agreement": float(results["trained_score"].mean()),
    "elapsed_minutes": elapsed_minutes(),
}

with open(EXPORT_DIR / "summary.json", "w") as file:
    json.dump(summary, file, indent=2)

print("\nResults:")
print(
    f"Baseline: {summary['baseline_agreement'] * 100:.2f}%"
)
print(
    f"Trained:  {summary['trained_agreement'] * 100:.2f}%"
)
print("Optimizer updates:", optimizer_updates)
print("Stopped by timer:", timer_stopped)
print(f"Elapsed: {elapsed_minutes():.1f} minutes")

display(results.head(10))

print("\nDiagnostic scores by answerability:")
display(
    results.groupby("answerable")[
        ["baseline_score", "trained_score"]
    ].mean()
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot([entry["loss"] for entry in history])
axes[0].set_title("Training loss")
axes[0].set_xlabel("Batch")
axes[0].set_ylabel("Loss")

axes[1].bar(
    ["Baseline", "Fine-tuned"],
    [
        summary["baseline_agreement"] * 100,
        summary["trained_agreement"] * 100,
    ],
)
axes[1].set_title("Validation subset: diagnostic agreement")
axes[1].set_ylabel("Score (%)")
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.savefig(
    EXPORT_DIR / "training_results.png",
    dpi=150,
)
plt.show()

print("\nTraining and evaluation complete.")
print("Run Block 3 to download the model and results.")

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

model_dir = Path("/content/visionbridge_t4/export/blip_vizwiz")
archive = Path("/content/visionbridge_model_only.zip")

assert (model_dir / "config.json").exists(), "Run training first."

with zipfile.ZipFile(
    archive, "w", compression=zipfile.ZIP_STORED
) as zip_file:
    for path in model_dir.rglob("*"):
        if path.is_file():
            zip_file.write(
                path,
                arcname=str(
                    Path("blip_vizwiz") / path.relative_to(model_dir)
                ),
            )

print(f"Model ZIP: {archive.stat().st_size / 1024**3:.2f} GB")
files.download(str(archive))